# МХТП, 7 семестр, вариант 10 – образец решения (альтернативный способ)
Выполнить ячейки по порядку (Среда выполнения → Выполнить все).

In [ ]:
# Ячейка 0. Библиотеки, константы, вспомогательные функции
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize, integrate

R = 8.314462618      # Дж/(моль∙К)
ATM = 101325.0       # Па
PSI = 6894.757       # Па
CAL = 4.184          # Дж


def quality(y, y_model, n_par):
    """Показатели адекватности модели: SSE, R2, скорректированный R2, s, max отн. ошибка, %."""
    y, y_model = np.asarray(y, float), np.asarray(y_model, float)
    e = y - y_model
    sse = float(e @ e)
    sst = float(((y - y.mean()) ** 2).sum())
    n = len(y)
    r2 = 1 - sse / sst
    return dict(SSE=sse, R2=r2, R2adj=1 - (1 - r2) * (n - 1) / (n - n_par),
                s=np.sqrt(sse / (n - n_par)), dmax=float(np.max(np.abs(e / y))) * 100)


FIG_DIR = None      # папка для сохранения рисунков; None - рисунки выводятся в конце
RESULTS = {}        # результаты по задачам (печатаются в последней ячейке)
FIGS = []
_TASK = [None]


def task(key, title):
    _TASK[0] = key
    RESULTS[key] = {"title": title, "lines": []}


def report(*args, sep=" "):
    RESULTS[_TASK[0]]["lines"].append(sep.join(str(a) for a in args))


def draw(fig, name):
    fig.tight_layout()
    if FIG_DIR is None:
        FIGS.append(fig)
    else:
        fig.savefig(f"{FIG_DIR}/{name}.png", dpi=200)
    plt.close(fig)


def show_results():
    for r in RESULTS.values():
        print("=" * 78)
        print(r["title"])
        print("=" * 78)
        print("\n".join(r["lines"]))
        print()


def show_figures():
    from IPython.display import display
    for fig in FIGS:
        display(fig)


plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.grid": True, "grid.alpha": 0.3})


In [ ]:
# Задача 1.1. Теплоёмкость пропана: полином 5-й степени по τ = T/1000
task("t11", "Задача 1.1")
T = np.array([50, 100, 150, 200, 273.15, 300, 400, 500, 600, 700,
              800, 900, 1000, 1100, 1200, 1300])
C = np.array([34.16, 41.3, 48.79, 56.07, 68.74, 73.93, 94.01, 112.59,
              128.7, 142.67, 154.77, 163.35, 174.6, 182.67, 189.74, 195.85])
deg, T0, T1 = 5, 50.0, 1300.0
tau = T / 1000                               # безразмерная температура
V = np.vander(tau, deg + 1, increasing=True)  # столбцы 1, τ, τ^2, ..., τ^5
c, *_ = np.linalg.lstsq(V, C, rcond=None)     # МНК через QR/SVD
cond_tau = np.linalg.cond(V)
cond_T = np.linalg.cond(np.vander(T, deg + 1, increasing=True))
C_mod = V @ c
q11 = quality(C, C_mod, deg + 1)
# интеграл полинома: ∫C dT = 1000·Σ c_j·τ^(j+1)/(j+1)
F = lambda t: 1000 * sum(cj * t ** (j + 1) / (j + 1) for j, cj in enumerate(c))
I_mod = F(T1 / 1000) - F(T0 / 1000)
I_trap = float(np.sum((C[1:] + C[:-1]) / 2 * np.diff(T)))   # трапеции вручную
I_simp = float(integrate.simpson(C, x=T))                    # Симпсон (неравном. сетка)
Cm = {k: v / (T1 - T0) for k, v in (("модель", I_mod), ("трапеции", I_trap), ("Симпсон", I_simp))}
for j, cj in enumerate(c):
    report(f"c{j} = {cj:10.4f} Дж/(моль·К)")
report(f"cond(V): по τ = {cond_tau:.3e}, по T = {cond_T:.3e}")
report(f"R2 = {q11['R2']:.6f}, s = {q11['s']:.3f} Дж/(моль·К), max δ = {q11['dmax']:.2f} %")
for k, v in Cm.items():
    report(f"<C>({k:9s}) = {v:.3f} Дж/(моль·К)")
res11 = dict(c=c, q=q11, I=dict(модель=I_mod, трапеции=I_trap, Симпсон=I_simp), Cm=Cm,
             cond_tau=cond_tau, cond_T=cond_T, e=C - C_mod, T=T.copy(), C=C.copy())

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
tg = np.linspace(T0, T1, 300) / 1000
ax[0].plot(T, C, "s", ms=4, label="таблица")
ax[0].plot(tg * 1000, np.vander(tg, deg + 1, increasing=True) @ c, label="полином 5-й степени")
ax[0].set(xlabel="T, К", ylabel="C, Дж/(моль∙К)")
ax[0].legend()
ax[1].stem(T, C - C_mod)
ax[1].set(xlabel="T, К", ylabel="Остаток, Дж/(моль∙К)")
draw(fig, "o11")


In [ ]:
# Задача 1.2. Модель y = a·(ln(x/b))^(-2/3): least_squares и две альтернативы
task("t12", "Задача 1.2")
x = np.array([0.263, 0.294, 0.321, 0.367, 0.429, 0.608, 1.33, 1.36])
y = np.array([0.34, 0.32, 0.24, 0.20, 0.16, 0.11, 0.06, 0.04])
f_given = lambda p, x: p[0] * np.log(x / p[1]) ** (-2 / 3)
sol = optimize.least_squares(lambda p: f_given(p, x) - y, x0=[0.1, 0.2],
                             bounds=([0, 1e-4], [np.inf, 0.2629]))
a, b = sol.x
q_given = quality(y, f_given(sol.x, x), 2)
# альтернатива 1: гипербола y = A/x + B (линейна по 1/x)
M = np.column_stack([1 / x, np.ones_like(x)])
(Ah, Bh), *_ = np.linalg.lstsq(M, y, rcond=None)
q_hyp = quality(y, Ah / x + Bh, 2)
# альтернатива 2: степенная y = K·x^(-n), МНК в логарифмах: ln y = ln K - n·ln x
slope, ln_K = np.polyfit(np.log(x), np.log(y), 1)
K_p, n_p = np.exp(ln_K), -slope
q_pow = quality(y, K_p * x ** (-n_p), 2)
report(f"заданная:  a = {a:.4f}, b = {b:.4f} кг/с; R2 = {q_given['R2']:.4f}, SSE = {q_given['SSE']:.5f}")
report(f"гипербола: A = {Ah:.4f} кг/с, B = {Bh:.4f}; R2 = {q_hyp['R2']:.4f}, SSE = {q_hyp['SSE']:.5f}")
report(f"степенная: K = {K_p:.4f}, n = {n_p:.4f}; R2 = {q_pow['R2']:.4f}, SSE = {q_pow['SSE']:.5f}")
res12 = dict(a=a, b=b, q=q_given, hyp=(Ah, Bh, q_hyp), pow=(K_p, n_p, q_pow),
             y_given=f_given(sol.x, x), x=x.copy(), y=y.copy())

fig, ax = plt.subplots(figsize=(6, 3.6))
xg = np.linspace(0.27, 1.4, 300)
ax.plot(x, y, "ko", label="опыт")
ax.plot(xg, f_given(sol.x, xg), label="заданная модель")
ax.plot(xg, Ah / xg + Bh, "--", label="гипербола")
ax.plot(xg, K_p * xg ** (-n_p), ":", label="степенная (МНК в логарифмах)")
ax.set(xlabel="x, кг/с", ylabel="y, доли")
ax.legend()
draw(fig, "o12")


In [ ]:
# Задача 1.3. Аррениус: линеаризация, МНК по относительным ошибкам, два участка
task("t13", "Задача 1.3")
t = np.arange(100, 501, 50.0)
k = np.array([0.10, 0.56, 1.76, 3.54, 9.53, 15.98, 25.04, 60.74, 143.60])
TK = t + 273.15
X = np.column_stack([np.ones_like(TK), -1 / (R * TK)])     # ln k = ln A - Ea/(R·T)
(lnA, Ea), *_ = np.linalg.lstsq(X, np.log(k), rcond=None)
q_ln = quality(np.log(k), X @ [lnA, Ea], 2)
# МНК по относительным отклонениям: (k_расч - k)/k -> min
rel = optimize.least_squares(lambda p: (np.exp(p[0] - p[1] / (R * TK)) - k) / k, x0=[lnA, Ea])
lnA_r, Ea_r = rel.x
q_rel = quality(k, np.exp(lnA_r - Ea_r / (R * TK)), 2)
# два участка: 100-300 °C и 300-500 °C
seg = {}
for name, msk in (("100–300 °C", t <= 300), ("300–500 °C", t >= 300)):
    (l_A, E), *_ = np.linalg.lstsq(X[msk], np.log(k[msk]), rcond=None)
    seg[name] = (E, np.exp(l_A), quality(np.log(k[msk]), X[msk] @ [l_A, E], 2)["R2"])
report(f"линеаризация: Ea = {Ea/1000:.2f} кДж/моль, A = {np.exp(lnA):.4e} 1/с, R2(ln k) = {q_ln['R2']:.4f}")
report(f"МНК по отн. ошибкам: Ea = {Ea_r/1000:.2f} кДж/моль, A = {np.exp(lnA_r):.4e} 1/с, "
       f"max δ = {q_rel['dmax']:.1f} %")
for name, (E, A_, r2_) in seg.items():
    report(f"участок {name}: Ea = {E/1000:.2f} кДж/моль, A = {A_:.4e} 1/с, R2(ln k) = {r2_:.4f}")
res13 = dict(Ea=Ea, A=np.exp(lnA), q_ln=q_ln, Ea_r=Ea_r, A_r=np.exp(lnA_r), q_rel=q_rel, seg=seg,
             e_ln=np.log(k) - X @ [lnA, Ea], t=t.copy(), k=k.copy())

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(1000 / TK, np.log(k), "ko", label="опыт")
ax.plot(1000 / TK, X @ [lnA, Ea], label="один участок")
for name, msk in (("100–300 °C", t <= 300), ("300–500 °C", t >= 300)):
    E, A_, _ = seg[name]
    ax.plot(1000 / TK[msk], np.log(A_) - E / (R * TK[msk]), "--", label=f"участок {name}")
ax.set(xlabel="1000/T, 1/К", ylabel="ln k (k в 1/с)")
ax.legend()
draw(fig, "o13")


In [ ]:
# Задача 1.4. Вязкость глицерина: полиномы по z = (T - 330)/60, F-критерий, ln μ(1/T)
task("t14", "Задача 1.4")
from scipy import stats
Tv = np.array([270, 280, 290, 300, 310, 320, 330, 350, 370, 390.0])   # при 250, 260 К данных нет
mu = np.array([12.00, 8.500, 6.050, 4.250, 3.500, 2.600, 2.000, 1.200, 0.730, 0.450])
z = (Tv - 330) / 60
fits, n = {}, len(mu)
for d in (1, 2, 3):
    Vd = np.vander(z, d + 1, increasing=True)
    bd, *_ = np.linalg.lstsq(Vd, mu, rcond=None)
    fits[d] = dict(b=bd, mu=Vd @ bd, q=quality(mu, Vd @ bd, d + 1))
for d in (2, 3):                              # нужен ли следующий член полинома?
    s1, s2 = fits[d - 1]["q"]["SSE"], fits[d]["q"]["SSE"]
    fits[d]["F"] = (s1 - s2) / (s2 / (n - d - 1))
    fits[d]["Fcr"] = stats.f.ppf(0.95, 1, n - d - 1)
# сравнение: ln μ = A + B/T + C/T^2 (уравнение Андраде с поправкой; линейный МНК по ln μ)
W = np.column_stack([np.ones_like(Tv), 1 / Tv, 1 / Tv ** 2])
pa, *_ = np.linalg.lstsq(W, np.log(mu), rcond=None)
q_and = quality(mu, np.exp(W @ pa), 3)
for d, f_ in fits.items():
    report(f"степень {d}: b = {np.round(f_['b'], 4)}")
    extra = f"; F = {f_['F']:.1f} (Fкр = {f_['Fcr']:.2f})" if "F" in f_ else ""
    report(f"   R2 = {f_['q']['R2']:.4f}, SSE = {f_['q']['SSE']:.3f}, "
           f"max δ = {f_['q']['dmax']:.1f} %{extra}")
report(f"ln mu = {pa[0]:.4f} + {pa[1]:.1f}/T + ({pa[2]:.4e})/T^2; R2 = {q_and['R2']:.5f}, "
       f"max δ = {q_and['dmax']:.2f} %")
res14 = dict(fits=fits, lnmu=(pa, q_and), T=Tv.copy(), mu=mu.copy())

fig, ax = plt.subplots(figsize=(6, 3.6))
Tg = np.linspace(270, 390, 300)
ax.semilogy(Tv, mu, "ko", label="опыт")
for d in (1, 2, 3):
    mg = np.vander((Tg - 330) / 60, d + 1, increasing=True) @ fits[d]["b"]
    ax.semilogy(Tg[mg > 0], mg[mg > 0], label=f"полином {d}-й степени")
ax.semilogy(Tg, np.exp(pa[0] + pa[1] / Tg + pa[2] / Tg ** 2), "k--", label="ln μ = A + B/T + C/T²")
ax.set(xlabel="T, К", ylabel="μ, мПа∙с (лог. шкала)")
ax.legend(fontsize=8)
draw(fig, "o14")


In [ ]:
# Задачи 2.1–2.2. Общая часть: RK, SRK, PR и метод Ньютона для Z
# Tc, К; Pc, атм; ω — справочник IFP Training, раздел A2
PROPS = {"H2": (33.2, 12.8, -0.22), "CH4": (190.6, 45.4, 0.008),
         "C2H6": (305.4, 48.2, 0.098), "C3H8": (369.8, 41.9, 0.152)}
# u, w в P = RT/(v - b) - a/(v^2 + u·b·v + w·b^2); Ωa, Ωb
EOS = {"RK": (1, 0, 0.42748, 0.08664), "SRK": (1, 0, 0.42748, 0.08664),
       "PR": (2, -1, 0.45724, 0.07780)}


def alpha(eos, Tr, w):
    if eos == "RK":
        return Tr ** -0.5
    m = (0.480 + 1.574 * w - 0.176 * w ** 2) if eos == "SRK" else (0.37464 + 1.54226 * w - 0.26992 * w ** 2)
    return (1 + m * (1 - np.sqrt(Tr))) ** 2


def z_newton(A, B, u, w, z0=1.0, tol=1e-12):
    """Корень кубического уравнения в Z методом Ньютона от газового приближения Z0 = 1."""
    c2, c1, c0 = -(1 + B - u * B), A + w * B ** 2 - u * B - u * B ** 2, -(A * B + w * B ** 2 + w * B ** 3)
    z, it = z0, 0
    while True:
        f, df = z ** 3 + c2 * z ** 2 + c1 * z + c0, 3 * z ** 2 + 2 * c2 * z + c1
        dz = f / df
        z, it = z - dz, it + 1
        if abs(dz) < tol:
            return z, it


def molar_volume(comp, y, T, P_atm, eos):
    P = P_atm * ATM
    u, w, Oa, Ob = EOS[eos]
    ai, bi = [], []
    for g in comp:
        Tc, Pc, om = PROPS[g]
        ai.append(Oa * (R * Tc) ** 2 / (Pc * ATM) * alpha(eos, T / Tc, om))
        bi.append(Ob * R * Tc / (Pc * ATM))
    ai, bi = np.array(ai), np.array(bi)
    sa = np.sqrt(ai)
    a, b = float((y @ sa) ** 2), float(y @ bi)          # правила смешения, kij = 0
    A, B = a * P / (R * T) ** 2, b * P / (R * T)
    z, it = z_newton(A, B, u, w)
    return dict(Z=z, v=z * R * T / P, a=a, b=b, A=A, B=B, it=it, ai=ai, bi=bi)


def table_volumes(comp, y, T, P_atm):
    vols = {"ИГ": dict(Z=1.0, v=R * T / (P_atm * ATM), it=0)}
    for eos in ("RK", "SRK", "PR"):
        vols[eos] = molar_volume(comp, y, T, P_atm, eos)
    report(f"{'модель':6s} {'Z':>8s} {'v, л/моль':>10s} {'итераций':>9s}")
    for k, d in vols.items():
        report(f"{k:6s} {d['Z']:8.4f} {d['v']*1000:10.4f} {d['it']:9d}")
    return vols


In [ ]:
# Задача 2.1. Водород: P = 50 атм, T = 400 К
task("t21", "Задача 2.1")
res21 = table_volumes(["H2"], np.array([1.0]), 400.0, 50.0)


In [ ]:
# Задача 2.2. CH4 25 %, C2H6 : C3H8 = 1 : 3; P = 120 атм, T = 420 К
task("t22", "Задача 2.2")
y22 = np.array([0.25, 0.75 / 4, 0.75 * 3 / 4])
res22 = table_volumes(["CH4", "C2H6", "C3H8"], y22, 420.0, 120.0)
Tpc = sum(yi * PROPS[g][0] for yi, g in zip(y22, ["CH4", "C2H6", "C3H8"]))
Ppc = sum(yi * PROPS[g][1] for yi, g in zip(y22, ["CH4", "C2H6", "C3H8"]))
report(f"Правило Кэя: Tpc = {Tpc:.1f} К, Ppc = {Ppc:.1f} атм; Tpr = {420/Tpc:.3f}, Ppr = {120/Ppc:.3f}")

# изотерма Z(P) смеси при 420 К по SRK и PR – где газ перестаёт быть идеальным
Pg = np.linspace(1, 200, 120)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
for eos in ("SRK", "PR"):
    ax[0].plot(Pg, [molar_volume(["H2"], np.array([1.0]), 400.0, p, eos)["Z"] for p in Pg], label=eos)
    ax[1].plot(Pg, [molar_volume(["CH4", "C2H6", "C3H8"], y22, 420.0, p, eos)["Z"] for p in Pg], label=eos)
ax[0].plot(50, res21["SRK"]["Z"], "ko")
ax[1].plot(120, res22["SRK"]["Z"], "ko")
ax[0].set(xlabel="P, атм", ylabel="Z", title="Водород, 400 К")
ax[1].set(xlabel="P, атм", ylabel="Z", title="CH₄–C₂H₆–C₃H₈, 420 К")
for a_ in ax:
    a_.axhline(1, color="gray", ls=":")
    a_.legend()
draw(fig, "o2")


In [ ]:
# Задачи 3.1–3.2. Общая часть: K-корреляция задания и метод Ньютона для Рачфорда–Райса
# A, B, C, D, форма: (1) ln K = -A/T^2 + B - C·ln P + D/P^2; (2) ... + D/P; T в °R, P в psia
KDATA = {"C2H4": (600076.9, 7.90595, 0.84677, 42.94594, 1),
         "CH4": (292860.0, 8.2445, 0.8951, 59.8465, 1),
         "C2H6": (687248.2, 7.90694, 0.866, 49.02654, 1),
         "C3H8": (970688.6, 7.15059, 0.76984, 6.90224, 2),
         "iC4": (1166846.0, 7.72668, 0.92213, 0.0, 1),
         "nC4": (1280557.0, 7.94986, 0.96455, 0.0, 1),
         "nC5": (1524891.0, 7.33129, 0.89143, 0.0, 1)}


def Kval(comp, T_K, P_Pa):
    TR, P = 1.8 * T_K, P_Pa / PSI
    A, B, Cc, D, form = (np.array([KDATA[c][i] for c in comp]) for i in range(5))
    return np.exp(-A / TR ** 2 + B - Cc * np.log(P) + np.where(form == 1, D / P ** 2, D / P))


def rr_newton(z, K, tol=1e-13):
    """Уравнение Рачфорда–Райса методом Ньютона (функция монотонна, старт e = 0.5)."""
    if z @ K <= 1:
        return 0.0, 0
    if z @ (1 / K) <= 1:
        return 1.0, 0
    lo, hi = 1 / (1 - K.max()), 1 / (1 - K.min())      # асимптоты функции
    e, it = 0.5, 0
    while True:
        d = 1 + e * (K - 1)
        f, df = np.sum(z * (K - 1) / d), -np.sum(z * (K - 1) ** 2 / d ** 2)
        e_new = e - f / df
        if not lo < e_new < hi:                  # страховка: шаг пополам к границе
            e_new = (e + (hi if e_new >= hi else lo)) / 2
        it += 1
        if abs(e_new - e) < tol:
            return min(max(e_new, 0.0), 1.0), it
        e = e_new


In [ ]:
# Задача 3.1. ТНК и ТКК смеси этилен 20 %, пропан 30 %, изобутан 30 %, н-бутан 20 % при 14 атм
task("t31", "Задача 3.1")
comp31 = ["C2H4", "C3H8", "iC4", "nC4"]
z31 = np.array([0.20, 0.30, 0.30, 0.20])


def bubble_dew(P_atm):
    P = P_atm * ATM
    # ln ΣKz = 0 и ln Σz/K = 0: логарифм делает функции почти линейными по T -> метод секущих
    fb = lambda T: np.log(z31 @ Kval(comp31, T, P))
    fd = lambda T: np.log(z31 @ (1 / Kval(comp31, T, P)))
    return optimize.newton(fb, 280.0), optimize.newton(fd, 330.0)


Tb, Td = bubble_dew(14.0)
Kb, Kd = Kval(comp31, Tb, 14 * ATM), Kval(comp31, Td, 14 * ATM)
report(f"P = 14 атм = {14*ATM/1e6:.4f} МПа = {14*ATM/PSI:.2f} psia")
report(f"ТНК = {Tb:.2f} К ({Tb-273.15:.2f} °C), ТКК = {Td:.2f} К ({Td-273.15:.2f} °C)")
report("y пузырька:", np.round(Kb * z31, 4), " x капли:", np.round(z31 / Kd, 4))
Pgrid = np.arange(4.0, 31.0, 1.0)
curve = np.array([bubble_dew(p) for p in Pgrid])
res31 = dict(Tb=Tb, Td=Td, Kb=Kb, Kd=Kd, Pgrid=Pgrid, curve=curve)

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(curve[:, 0] - 273.15, Pgrid, label="линия кипения (ТНК)")
ax.plot(curve[:, 1] - 273.15, Pgrid, label="линия росы (ТКК)")
ax.plot([Tb - 273.15, Td - 273.15], [14, 14], "ko-", label="14 атм")
ax.set(xlabel="t, °C", ylabel="P, атм")
ax.legend()
draw(fig, "o31")


In [ ]:
# Задача 3.2. Сепараторы: базовый режим (С1 9.8 МПа/40.8 °C, С2 8.7/12.8, С3 8.7/-40.8)
task("t32", "Задача 3.2 (базовый режим)")
comp32 = ["CH4", "C2H6", "C3H8", "iC4", "nC4", "nC5"]
zF = np.array([0.72211, 0.03608, 0.22363, 0.00646, 0.00466, 0.00706])
STAGE = {"С1": (9.8e6, 40.8), "С2": (8.7e6, 12.8), "С3": (8.7e6, -40.8)}


def run_block(stages, F=100.0):
    """Цепочка сепараторов на 100 моль/ч сырья; возвращает ступени и покомпонентные потоки."""
    feed, result, liq = F * zF, [], np.zeros(len(zF))
    for s in stages:
        P, t = STAGE[s]
        Fs = feed.sum()
        zs = feed / Fs
        K = Kval(comp32, t + 273.15, P)
        e, it = rr_newton(zs, K)
        x = zs / (1 + e * (K - 1))
        yv = K * x if e > 0 else np.zeros_like(zs)
        x = x if e < 1 else np.zeros_like(zs)
        result.append(dict(name=s, e=e, it=it, K=K, x=x, y=yv, F=Fs))
        liq += Fs * (1 - e) * x
        feed = Fs * e * yv
    return result, feed, liq                        # feed – товарный газ, моль/ч по компонентам


base, gas, liq = run_block(["С1", "С2", "С3"])
for s in base:
    report(f"{s['name']}: e = {s['e']:.5f} ({s['it']} итераций Ньютона), питание {s['F']:.3f} моль/ч")
    report("   x:", " ".join(f"{v:.5f}" for v in s["x"]))
    report("   y:", " ".join(f"{v:.5f}" for v in s["y"]))
report(f"Газ: {gas.sum():.3f} моль/ч; жидкость: {liq.sum():.3f} моль/ч; "
       f"баланс: {100 - gas.sum() - liq.sum():.1e}")
res32 = dict(base=base, gas=gas, liq=liq)


In [ ]:
# Задача 3.2. Исследование: влияние количества сепараторов (1, 2 или 3)
task("t32_study", "Задача 3.2 (исследование числа сепараторов)")
SCHEMES = [("1", ["С1"]), ("2", ["С1", "С2"]), ("3", ["С1", "С2", "С3"]),     # ряд А: С1 -> ...
           ("1", ["С3"]), ("2", ["С2", "С3"]), ("3", ["С1", "С2", "С3"])]     # ряд Б: ... -> С3
study = []
for nst, st in SCHEMES:
    _, g, l = run_block(st)
    G = g.sum()
    study.append(dict(n=int(nst), stages=" + ".join(st), G=G,
                      CH4_gas=g[0] / (100 * zF[0]) * 100,                     # извлечение метана в газ, %
                      C3p_gas=g[2:].sum() / G * 100 if G > 0 else np.nan,     # C3+ в газе, %
                      C3_liq=l[2] / (100 * zF[2]) * 100))                     # пропан в жидкость, %
report(f"{'схема':16s} {'газ, моль/ч':>11s} {'CH4 в газ, %':>13s} {'C3+ в газе, %':>14s} {'C3H8 в жидк., %':>16s}")
for r_ in study:
    c3 = "—" if np.isnan(r_["C3p_gas"]) else f"{r_['C3p_gas']:.3f}"
    report(f"{r_['n']}: {r_['stages']:13s} {r_['G']:11.3f} {r_['CH4_gas']:13.2f} {c3:>14s} {r_['C3_liq']:16.2f}")
res32["study"] = study

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
for lab, part in (("ряд А: С1 → … (отключаются последние)", study[:3]),
                  ("ряд Б: … → С3 (отключаются первые)", study[3:])):
    ax[0].plot([r_["n"] for r_ in part], [r_["CH4_gas"] for r_ in part], "o-", label=lab)
    ax[1].plot([r_["n"] for r_ in part], [r_["C3p_gas"] for r_ in part], "o-", label=lab)
ax[0].set(xlabel="Число сепараторов", ylabel="Извлечение CH₄ в газ, %", xticks=[1, 2, 3])
ax[1].set(xlabel="Число сепараторов", ylabel="C₃₊ в газе, %", xticks=[1, 2, 3])
ax[0].legend(fontsize=7)
draw(fig, "o32")


In [ ]:
# Задача 4.1. Синтез аммиака: уравнение равновесия как полином по степени полноты ξ
task("t41", "Задача 4.1")
Kp, P, T = 0.05, 220.0, 589.0                # 1/атм, атм, К
nN, nH, nA = 1.1, 3.0, 0.2                   # моль/ч на входе
# yA/(yN^0.5·yH^1.5) = Kp·P;  n = 4.3 - ξ  ->  (nA + ξ)^2·n^2 = (Kp·P)^2·(nN - ξ/2)(nH - 3ξ/2)^3
p = np.polynomial.Polynomial
xi_ = p([0, 1])
lhs = (nA + xi_) ** 2 * (nN + nH + nA - xi_) ** 2
rhs = (Kp * P) ** 2 * (nN - xi_ / 2) * (nH - 1.5 * xi_) ** 3
roots = (lhs - rhs).roots()
xi = float([r.real for r in roots if abs(r.imag) < 1e-9 and 0 < r.real < 2][0])
n41 = np.array([nN - xi / 2, nH - 1.5 * xi, nA + xi])
y41 = n41 / n41.sum()
report("корни полинома 4-й степени:", np.round(roots, 4))
report(f"физический корень ξ = {xi:.6f} моль/ч (0 < ξ < 2)")
for nm, a, b, c in zip(("N2", "H2", "NH3"), (nN, nH, nA), n41, y41):
    report(f"{nm:4s} {a:.4f} -> {b:.4f} моль/ч, y = {c:.4f}")
report(f"проверка: Kp = {y41[2] / (y41[0] ** 0.5 * y41[1] ** 1.5) / P:.6f} 1/атм")
# влияние давления на равновесный выход NH3 (Kp при 589 К постоянна)
Pg = np.linspace(20, 400, 60)
yA = []
for Pi in Pg:
    rr = ((nA + xi_) ** 2 * (4.3 - xi_) ** 2 - (Kp * Pi) ** 2 * (nN - xi_ / 2) * (nH - 1.5 * xi_) ** 3).roots()
    x_ = [r.real for r in rr if abs(r.imag) < 1e-9 and -0.2 < r.real < 2][0]
    yA.append((nA + x_) / (4.3 - x_))
res41 = dict(xi=xi, n=n41, y=y41, roots=roots, Pg=Pg, yA=np.array(yA))

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(Pg, np.array(yA) * 100)
ax.plot(P, y41[2] * 100, "ko")
ax.set(xlabel="P, атм", ylabel="Равновесная доля NH₃, % (589 К)")
draw(fig, "o41")


In [ ]:
# Задача 4.2а. C2H6 <=> C2H4 + H2 при 1100 К: K по уравнению Кирхгофа от 298 К
task("t42a", "Задача 4.2а")
# ΔH°f, ΔG°f (298 К), кал/моль – справочник IFP, E18, E19; Cp = a + bT + cT^2 + dT^3, кал/(моль·К) – D1
dHf = {"C2H6": -20236.0, "C2H4": 12496.0, "H2": 0.0}
dGf = {"C2H6": -7860.0, "C2H4": 16282.0, "H2": 0.0}
cp = {"C2H6": (1.648, 4.124e-2, -1.53e-5, 1.74e-9),
      "C2H4": (0.944, 3.735e-2, -1.993e-5, 4.22e-9),
      "H2": (6.483, 0.221e-2, -0.333e-5, 1.826e-9)}
nu = {"C2H4": 1, "H2": 1, "C2H6": -1}
dCp = lambda T: sum(v * (cp[s][0] + cp[s][1] * T + cp[s][2] * T ** 2 + cp[s][3] * T ** 3) for s, v in nu.items())
dH298 = sum(v * dHf[s] for s, v in nu.items()) * CAL               # Дж/моль
dG298 = sum(v * dGf[s] for s, v in nu.items()) * CAL
dS298 = (dH298 - dG298) / 298.15


def K_kirchhoff(T):
    dH = dH298 + integrate.quad(dCp, 298.15, T)[0] * CAL
    dS = dS298 + integrate.quad(lambda x: dCp(x) / x, 298.15, T)[0] * CAL
    dG = dH - T * dS
    return np.exp(-dG / (R * T)), dH, dG


K1000_table = np.exp(-(28249 - 26130) * CAL / (R * 1000))   # по ΔG°f при 1000 К из таблицы
K1000, _, _ = K_kirchhoff(1000.0)
K42, dH1100, dG1100 = K_kirchhoff(1100.0)
x42 = np.sqrt(K42 / (K42 + 1.0))                              # P = 1 атм
n42 = np.array([1 - x42, x42, x42])
y42 = n42 / n42.sum()
report(f"ΔH°298 = {dH298/1000:.2f} кДж/моль, ΔG°298 = {dG298/1000:.2f} кДж/моль, ΔS°298 = {dS298:.2f} Дж/(моль·К)")
report(f"проверка при 1000 К: K(Кирхгоф) = {K1000:.4f}, K(таблица ΔG°f) = {K1000_table:.4f}")
report(f"1100 К: ΔH° = {dH1100/1000:.2f} кДж/моль, ΔG° = {dG1100/1000:.3f} кДж/моль, K = {K42:.4f}")
report(f"x = {x42:.5f}; y: C2H6 {y42[0]:.4f}, C2H4 {y42[1]:.4f}, H2 {y42[2]:.4f}")
Tg = np.linspace(300, 1150, 60)
res42 = dict(dH298=dH298, dG298=dG298, dS298=dS298, K1000=K1000, K1000_table=K1000_table,
             K=K42, dH1100=dH1100, dG1100=dG1100, x=x42, n=n42, y=y42,
             curve=np.array([K_kirchhoff(t_)[2] for t_ in Tg]), Tg=Tg)

fig, ax = plt.subplots(figsize=(6, 3.4))
Tt = np.array([298, 400, 600, 800, 1000.0])
dGt = (np.array([16282, 17675, 20918, 24490, 28249]) - np.array([-7860, -3447, 5970, 15920, 26130])) * CAL
ax.plot(Tg, res42["curve"] / 1000, label="Кирхгоф (Cp из D1)")
ax.plot(Tt, dGt / 1000, "ks", label="таблица E19")
ax.plot(1100, dG1100 / 1000, "ro", label="1100 К")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="T, К", ylabel="ΔG°r, кДж/моль")
ax.legend()
draw(fig, "o42")


In [ ]:
# Задача 4.2б. Две реакции при 1367 К: сведение системы к одному уравнению
task("t42b", "Задача 4.2б")
K1, K2 = 37.6, 0.769                          # P = 1 атм


def x2_of_x1(x1):
    """Из K2 = x2·(x1 + x2)/((x1 - x2)·(1 + x1 + x2)) — квадратное уравнение относительно x2."""
    a = 1 + K2
    b = x1 + K2                              # (1 + K2)·x2^2 + (x1 + K2)·x2 - K2·x1·(1 + x1) = 0
    c = -K2 * x1 * (1 + x1)
    return (-b + np.sqrt(b * b - 4 * a * c)) / (2 * a)


def g(x1):
    x2 = x2_of_x1(x1)
    n_ = np.array([1 - x1, x1 - x2, x1 + x2, x2])
    y_ = n_ / n_.sum()
    return y_[1] * y_[2] - K1 * y_[0]


x1 = optimize.brentq(g, 1e-9, 1 - 1e-12, xtol=1e-14)
x2 = x2_of_x1(x1)
n43 = np.array([1 - x1, x1 - x2, x1 + x2, x2])
y43 = n43 / n43.sum()
report(f"x1 = {x1:.5f}, x2 = {x2:.5f}")
for nm, a, b in zip(("C2H6", "C2H4", "H2", "C2H2"), n43, y43):
    report(f"{nm:5s} {a:.5f} моль  y = {b:.4f}")
report(f"проверка: K1 = {y43[1]*y43[2]/y43[0]:.4f}, K2 = {y43[3]*y43[2]/y43[1]:.5f}")
res43 = dict(x1=x1, x2=x2, n=n43, y=y43)


In [ ]:
# Задача 5.1. Три колонны: метод Гаусса с выбором главного элемента (собственная реализация)
task("t51", "Задача 5.1")
def gauss(A, b):
    A, b = np.array(A, float), np.array(b, float)
    n = len(b)
    for i in range(n):                                   # прямой ход
        p_ = i + np.argmax(np.abs(A[i:, i]))
        A[[i, p_]], b[[i, p_]] = A[[p_, i]], b[[p_, i]]
        for j in range(i + 1, n):
            f = A[j, i] / A[i, i]
            A[j, i:] -= f * A[i, i:]
            b[j] -= f * b[i]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):                       # обратный ход
        x[i] = (b[i] - A[i, i + 1:] @ x[i + 1:]) / A[i, i]
    return x


comps = ["ксилол", "стирол", "толуол", "бензол"]
F = 1000.0
xF = np.array([15, 25, 40, 20]) / 100
xD1, xB1 = np.array([7, 4, 54, 35]) / 100, np.array([18, 24, 42, 16]) / 100
xD2, xB2 = np.array([15, 10, 54, 21]) / 100, np.array([24, 65, 10, 1]) / 100
D1, B1, D2, B2 = gauss(np.column_stack([xD1, xB1, xD2, xB2]), F * xF)
D, B = D1 + B1, D2 + B2
xD, xB = (D1 * xD1 + B1 * xB1) / D, (D2 * xD2 + B2 * xB2) / B
# степень извлечения компонента колонной №1 в дистиллят: D·xD,i / (F·xF,i)
rec1 = D * xD / (F * xF) * 100
streams = {"F": (F, xF), "D": (D, xD), "B": (B, xB), "D1": (D1, xD1), "B1": (B1, xB1),
           "D2": (D2, xD2), "B2": (B2, xB2)}
report("поток   расход, моль/ч   " + "  ".join(f"{c:>7s}" for c in comps))
for k, (Fl, xx) in streams.items():
    report(f"{k:5s} {Fl:12.2f}       " + "  ".join(f"{v:7.4f}" for v in xx))
report("извлечение в D колонны №1, %:", "  ".join(f"{c} {v:.1f}" for c, v in zip(comps, rec1)))
report(f"проверка общего баланса: F - (D1 + B1 + D2 + B2) = {F - D1 - B1 - D2 - B2:.2e} моль/ч")
res51 = dict(streams=streams, rec1=rec1)


In [ ]:
# Задача 5.2. Процесс Toray: последовательный модульный расчёт, разрыв по рециклу (поток 7)
task("t52", "Задача 5.2")
F1, conv = 100.0, 0.58          # свежий толуол, моль/ч; конверсия толуола за проход
# компоненты: [толуол, бензол, ксилол]


def flowsheet(s7):
    """Один проход по схеме при заданном рецикле s7; возвращает все потоки."""
    s1 = np.array([F1, 0, 0])
    s2 = s1 + s7                                        # смеситель M
    xi = conv * s2[0] / 2                               # реактор R: 2 C7H8 -> C6H6 + C8H10
    sR = np.array([-2 * xi, xi, xi])
    s3 = s2 + sR
    s4 = np.array([0.001 * s3[0], 0.995 * s3[1], 0.0])  # S1: бензольная фракция
    s5 = s3 - s4
    s7_new = np.array([0.995 * s5[0], s5[1], 0.001 * s5[2]])   # S2: рецикл
    s6 = s5 - s7_new
    return dict(s1=s1, s2=s2, R=sR, s3=s3, s4=s4, s5=s5, s6=s6, s7=s7_new, xi=xi)


def solve_recycle(method, tol=1e-10, maxit=500):
    s7, hist = np.zeros(3), []
    g_prev = x_prev = None
    for it in range(1, maxit + 1):
        g = flowsheet(s7)["s7"]                          # g(x): новое значение рецикла
        err = np.max(np.abs(g - s7))
        hist.append((it, s7[0], err))
        if err < tol:
            return flowsheet(s7), hist
        if method == "Вегстейн" and g_prev is not None:  # ускорение по каждому компоненту
            dx = s7 - x_prev
            sl = np.divide(g - g_prev, dx, out=np.zeros(3), where=np.abs(dx) > 1e-14)
            q = np.clip(sl / (sl - 1), -5, 0)
            x_new = q * s7 + (1 - q) * g
        else:
            x_new = g                                    # простая итерация
        x_prev, g_prev, s7 = s7, g, x_new
    raise RuntimeError("нет сходимости")


sol = {}
for meth in ("простая итерация", "Вегстейн"):
    sol[meth] = solve_recycle(meth)
    report(f"{meth}: {len(sol[meth][1])} итераций")
S = sol["Вегстейн"][0]
report(f"{'':8s}" + "".join(f"{k:>9s}" for k in ("1", "2", "R", "3", "4", "5", "6", "7")))
for i, c in enumerate(("Толуол", "Бензол", "Ксилол")):
    report(f"{c:8s}" + "".join(f"{S[k][i]:9.3f}" for k in ("s1", "s2", "R", "s3", "s4", "s5", "s6", "s7")))
report(f"{'Всего':8s}" + "".join(f"{S[k].sum():9.3f}" for k in ("s1", "s2", "R", "s3", "s4", "s5", "s6", "s7")))
X_xyl = S["s6"][2] / F1
X_tot = 1 - (S["s4"][0] + S["s6"][0]) / F1
Y_benz = S["s4"][1] / F1                                 # выход бензола на свежий толуол
report(f"конверсия толуола в ксилол {X_xyl*100:.2f} %, общая конверсия толуола {X_tot*100:.2f} %")
report(f"выход бензола {Y_benz*100:.2f} % (моль на моль свежего толуола)")
res52 = dict(S=S, hist={m: h for m, (_, h) in sol.items()}, X_xyl=X_xyl, X_tot=X_tot, Y_benz=Y_benz)

fig, ax = plt.subplots(figsize=(6, 3.4))
for meth, (_, h) in sol.items():
    h = np.array(h)
    ax.semilogy(h[:, 0], h[:, 2], "o-", ms=3, label=meth)
ax.set(xlabel="Номер итерации", ylabel="Невязка рецикла, моль/ч")
ax.legend()
draw(fig, "o52")


## Результаты расчётов

In [ ]:
# Результаты всех задач
show_results()


In [ ]:
# Графики всех задач
show_figures()
